# PGR Safety Extension: Hazard Compression in Diffusion-Based Replay

This notebook runs the full experiment pipeline for our safety extension to
Prioritized Generative Replay (Wang et al., ICLR 2025).

**Requirements:** A100 GPU runtime on Colab Pro.

**Experiments:**
- SAC baseline (no diffusion)
- PGR (original, with hazard cost tracking + DiffHz probe)
- PGR+Memory (our contribution: rare-event buffer + Lagrangian constraint)

## 1. Setup

In [ ]:
# Verify GPU
!nvidia-smi

In [ ]:
# Clone PGR repo
!git clone https://github.com/renwang435/pgr.git
%cd pgr
!git submodule update --init --recursive

In [ ]:
# Upload safety extension files
# Option 1: Upload from local machine
# from google.colab import files
# Upload safety/ directory

# Option 2: Copy from Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/pgr_safety/safety .

# Option 3: Create files inline (run colab_setup.py after creating safety/)
!mkdir -p safety

In [ ]:
# Install dependencies
!apt-get -qq update && apt-get -qq install -y libosmesa6-dev libgl1-mesa-glx libglfw3 patchelf

!pip install -q torch numpy accelerate einops ema-pytorch tqdm gin-config
!pip install -q mujoco dm-control dm-env dm-tree
!pip install -q "gym==0.23.0"
!pip install -q git+https://github.com/conglu1997/dmcgym.git@812905790dd87a448c9544a0beccb8b05ea2a850
!pip install -e synther/REDQ
!pip install -q h5py sortedcontainers pyrallis matplotlib

In [ ]:
# Patch d4rl and ipdb imports (not needed for online training)
!sed -i 's/^import d4rl/# import d4rl/' synther/diffusion/utils.py
!sed -i 's/^from ipdb import/# from ipdb import/' synther/diffusion/utils.py synther/diffusion/denoiser_network_cond.py

In [ ]:
# Verify setup
import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

import gym, dmcgym
env = gym.make('cheetah-run-v0')
obs = env.reset()
print(f'cheetah-run-v0: obs_dim={obs.shape[0]}, act_dim={env.action_space.shape[0]}')
env.close()
print('Setup OK!')

## 2. Quick Smoke Test (1K steps)

Verify everything works before committing to long runs.

In [ ]:
!python safety/online_cost_cond.py \
    --env cheetah-run-v0 \
    --seed 42 \
    --gin_config_files config/online/sac_cond_synther_dmc.gin \
    --gin_params 'redq_sac.cond_top_frac = 0.25' 'redq_sac.epochs = 2' \
    --velocity_threshold 7.0 \
    --results_folder ./smoke_test

## 3. Run Experiments

Each run is ~30-90 minutes on A100. Total: ~5-14 hours for all 9 runs.

In [ ]:
RESULTS_DIR = './safety_results'
ENV = 'cheetah-run-v0'
GIN_CONFIG = 'config/online/sac_cond_synther_dmc.gin'
VEL_THRESHOLD = 7.0
SEEDS = [42, 123, 456]

In [ ]:
# SAC baselines (3 seeds)
for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'SAC baseline, seed={seed}')
    print(f'{"="*60}')
    !python safety/online_cost_cond.py \
        --env {ENV} --seed {seed} \
        --gin_config_files config/online/sac.gin \
        --gin_params 'redq_sac.disable_diffusion = True' \
        --velocity_threshold {VEL_THRESHOLD} \
        --results_folder {RESULTS_DIR}/sac_seed{seed}

In [ ]:
# PGR (3 seeds)
for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'PGR, seed={seed}')
    print(f'{"="*60}')
    !python safety/online_cost_cond.py \
        --env {ENV} --seed {seed} \
        --gin_config_files {GIN_CONFIG} \
        --gin_params 'redq_sac.cond_top_frac = 0.25' \
        --velocity_threshold {VEL_THRESHOLD} \
        --results_folder {RESULTS_DIR}/pgr_seed{seed}

In [ ]:
# PGR+Memory (3 seeds)
for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'PGR+Memory, seed={seed}')
    print(f'{"="*60}')
    !python safety/online_cost_cond.py \
        --env {ENV} --seed {seed} \
        --gin_config_files {GIN_CONFIG} \
        --gin_params 'redq_sac.cond_top_frac = 0.25' \
        --velocity_threshold {VEL_THRESHOLD} \
        --use_rare_buffer --use_lagrangian \
        --results_folder {RESULTS_DIR}/pgr_memory_seed{seed}

## 4. Generate Figures and Table

In [ ]:
!python safety/make_figures.py --results_dir ./safety_results --output_dir ./figures

In [ ]:
# Display figures inline
from IPython.display import Image, display
import os

for fig in ['figure1_diffhz.pdf', 'figure2_curves.pdf']:
    path = f'./figures/{fig}'
    if os.path.exists(path):
        # Convert PDF to PNG for display
        png_path = path.replace('.pdf', '.png')
        !convert -density 150 {path} {png_path}
        if os.path.exists(png_path):
            display(Image(png_path))

## 5. Save Results to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/pgr_safety_results
!cp -r ./safety_results /content/drive/MyDrive/pgr_safety_results/
!cp -r ./figures /content/drive/MyDrive/pgr_safety_results/
print('Results saved to Google Drive!')

## 6. Optional: Two-Phase Experiment (Experiment C)

In [ ]:
# PGR two-phase (1 seed first)
!python safety/online_cost_cond.py \
    --env cheetah-run-v0 --seed 42 \
    --gin_config_files {GIN_CONFIG} \
    --gin_params 'redq_sac.cond_top_frac = 0.25' \
    --velocity_threshold 7.0 \
    --two_phase \
    --results_folder ./safety_results/pgr_twophase_seed42

In [ ]:
# PGR+Memory two-phase (1 seed)
!python safety/online_cost_cond.py \
    --env cheetah-run-v0 --seed 42 \
    --gin_config_files {GIN_CONFIG} \
    --gin_params 'redq_sac.cond_top_frac = 0.25' \
    --velocity_threshold 7.0 \
    --use_rare_buffer --use_lagrangian \
    --two_phase \
    --results_folder ./safety_results/pgr_memory_twophase_seed42